In [1]:
!pip install pandas scikit-learn transformers torch


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import pandas as pd
import pickle

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

from transformers import pipeline

In [6]:
import pandas as pd

software_df = pd.read_json("dataset.json")
nonsoftware_df = pd.read_json("non_related.json")

software_df.head()

,id,prompt,skill,difficulty
0,1,design chat application backend,file_handling,medium
1,2,implement kruskal algorithm,devops,medium
2,3,design notification system,data_structures,medium
3,4,implement jwt authentication,devops,hard
4,5,implement kruskal algorithm,devops,medium


In [7]:
software_df["text"] = software_df["prompt"]
software_df["label"] = "software"

software_df = software_df[["text","label"]]

In [8]:
nonsoftware_df["text"] = nonsoftware_df["prompt"]
nonsoftware_df["label"] = "not_scope"

nonsoftware_df = nonsoftware_df[["text","label"]]

In [9]:
dataset = pd.concat([software_df, nonsoftware_df], ignore_index=True)

dataset.head()

,text,label
0,design chat application backend,software
1,implement kruskal algorithm,software
2,design notification system,software
3,implement jwt authentication,software
4,implement kruskal algorithm,software


In [10]:
dataset["label"].value_counts()

label
software     2000
not_scope     500
Name: count, dtype: int64

In [12]:
nonsoftware_df = nonsoftware_df.sample(2000, replace=True)
dataset = pd.concat([software_df, nonsoftware_df], ignore_index=True)

In [13]:
dataset = dataset.drop_duplicates(subset="text")

In [14]:
dataset = dataset.sample(frac=1).reset_index(drop=True)

In [15]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    dataset["text"],
    dataset["label"],
    test_size=0.2,
    random_state=42
)

In [16]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer()

X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

In [17]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)

model.fit(X_train_vec, y_train)

LogisticRegression(max_iter=1000)

In [18]:
from sklearn.metrics import accuracy_score

pred = model.predict(X_test_vec)

print("Accuracy:", accuracy_score(y_test, pred))

Accuracy: 0.9


In [19]:
import pickle

pickle.dump(model, open("scope_model.pkl","wb"))
pickle.dump(vectorizer, open("vectorizer.pkl","wb"))

In [20]:
def test_query(q):
    vec = vectorizer.transform([q])
    print("Prediction:", model.predict(vec)[0])

In [22]:
test_query("implement binary search in python")

Prediction: software


In [23]:
test_query("best pizza recipe")

Prediction: not_scope


In [24]:
pip install accelerate


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
